# Industry Classification Experiment
## Worth AI Production Pipeline vs Consensus Level 2 XGBoost

**How to use this notebook:**

1. Run the experiment script first (generates all data files):
   ```bash
   # With Redshift (from inside VPC or with public access enabled):
   python modeling/experiment.py --limit 10000

   # Without Redshift (synthetic data, same schema):
   python modeling/experiment.py --synthetic --limit 5000
   ```
2. Then run all cells in this notebook: **Cell → Run All**

---

### What is being compared

| | Scenario A — Production | Scenario B — Consensus |
|---|---|---|
| **Level 1 model** | `entity_matching_20250127 v1` | Same — identical |
| **Classification** | SQL rule: `max(zi_conf, efx_conf)` wins | Level 2 XGBoost `multi:softprob` |
| **Output** | 1 NAICS code, no probability | Top-3 codes with calibrated probability |
| **UK SIC** | Received from OC, silently dropped | Primary output for GB companies |
| **AML signals** | 0 — never produced | 6 signal types |
| **KYB** | Never produced | APPROVE / REVIEW / ESCALATE / REJECT |
| **Jurisdiction routing** | Always NAICS | Correct taxonomy per country |

### What the columns mean

**Level 1 columns** (`L1:`) — outputs of the entity matching XGBoost model, stored in Redshift match tables:
- `L1: EFX confidence` → `efx_matches_custom_inc_ml.efx_probability`
- `L1: OC confidence` → `oc_matches_custom_inc_ml.oc_probability`
- `L1: ZI confidence` → `zoominfo_matches_custom_inc_ml.zi_probability`
- `L1: Liberty confidence` → `{TABLE}_results.parquet` (local batch output)

**Production columns** (`Prod:`) — what `customer_table.sql` currently stores:
- `Prod: NAICS code` → `datascience.customer_files.naics_code`
- `Prod: Winning source` → always `zoominfo` or `equifax` (rule only compares these two)
- `Prod: UK SIC returned` → OC returns it; production drops it silently

**Consensus columns** (`Cons:`) — Level 2 XGBoost output (not yet in production):
- `Cons: Primary code` → top-1 prediction from `multi:softprob`
- `Cons: Probability` → calibrated confidence 0–100%
- `Cons: KYB recommendation` → APPROVE / REVIEW / ESCALATE / REJECT
- `Cons: Risk flags` → AML signal types detected
- `Cons: Majority agreement` → fraction of sources returning the same code


In [ ]:
import os, sys, json, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter
from pathlib import Path

warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path('.').resolve()))

# ── Style ─────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0F1117', 'axes.facecolor': '#1A1F2E',
    'axes.edgecolor': '#2D3748', 'axes.labelcolor': '#E2E8F0',
    'xtick.color': '#A0AEC0', 'ytick.color': '#A0AEC0',
    'text.color': '#E2E8F0', 'grid.color': '#2D3748', 'grid.alpha': 0.4,
    'font.size': 11, 'axes.titlesize': 12, 'axes.titleweight': 'bold',
})
BLUE='#4299E1'; GREEN='#48BB78'; RED='#FC8181'; GOLD='#ECC94B'
GRAY='#718096'; PURPLE='#9F7AEA'
SRC_C = {'equifax':BLUE,'open_corporates':GREEN,'zoominfo':GOLD,'liberty':RED}
KYB_C = {'APPROVE':GREEN,'REVIEW':BLUE,'ESCALATE':GOLD,'REJECT':RED}

def _save(name):
    plt.savefig(f'data/modeling/{name}', dpi=150, bbox_inches='tight', facecolor='#0F1117')
    plt.show()

print('Libraries loaded.')

In [ ]:
# ── Load experiment outputs produced by modeling/experiment.py ─────────────────
RESULTS_CSV   = Path('experiment_results.csv')
L1_REPORT     = Path('data/modeling/l1_report.json')
EVAL_REPORT   = Path('data/modeling/evaluation_report.json')

if not RESULTS_CSV.exists():
    print("⚠️  experiment_results.csv not found.")
    print("Run first:  python modeling/experiment.py --synthetic")
    print("Or with real data:  python modeling/experiment.py --limit 10000")
    raise SystemExit(0)

df        = pd.read_csv(RESULTS_CSV)
l1_report = json.load(open(L1_REPORT))  if L1_REPORT.exists()  else {}
eval_b    = json.load(open(EVAL_REPORT)) if EVAL_REPORT.exists() else {}
n         = len(df)

data_src  = df['_data_source'].iloc[0] if '_data_source' in df.columns else 'unknown'
is_real   = 'REAL' in str(data_src).upper()

print(f"Loaded {n:,} companies")
print(f"Data source:  {data_src}")
print(f"Real Redshift data: {is_real}")
if not is_real:
    print()
    print("NOTE: Synthetic data mirrors the exact Redshift schema.")
    print("To use real data: enable Redshift public access or connect via VPN,")
    print("then run:  python modeling/experiment.py --limit 10000")

---
## Part 1 — Level 1: Entity Matching Source Confidences

The Level 1 XGBoost model (`entity_matching_20250127 v1`) already runs in production.
It produces a match confidence 0–1 for each source: Equifax, OpenCorporates, ZoomInfo, Liberty.

**These confidence scores already exist in Redshift** — in the three match tables:
- `datascience.efx_matches_custom_inc_ml.efx_probability`
- `datascience.oc_matches_custom_inc_ml.oc_probability`
- `datascience.zoominfo_matches_custom_inc_ml.zi_probability`

The Level 1 model uses **33 pairwise text/address similarity features** per candidate pair —
Jaccard k-grams on business name, street name, short name (city-stripped), plus exact matches
on zip code, city, street number and block.

In [ ]:
# ── Source confidence distributions ───────────────────────────────────────────
l1_cols = {
    'L1: EFX confidence':     ('Equifax',          BLUE),
    'L1: OC confidence':      ('OpenCorporates',   GREEN),
    'L1: ZI confidence':      ('ZoomInfo',         GOLD),
    'L1: Liberty confidence': ('Liberty',          RED),
}
available_l1 = {k: v for k, v in l1_cols.items() if k in df.columns}

fig, axes = plt.subplots(1, len(available_l1), figsize=(5*len(available_l1), 4))
if len(available_l1) == 1: axes = [axes]

for ax, (col, (label, colour)) in zip(axes, available_l1.items()):
    vals = pd.to_numeric(df[col], errors='coerce').dropna()
    matched = (vals >= 0.80).sum()
    ax.hist(vals, bins=25, color=colour, alpha=0.85, edgecolor='#0F1117')
    ax.axvline(0.80, color='white', linestyle='--', linewidth=1.5, label='0.80 threshold')
    ax.set_title(f'{label}\n{matched}/{n} matched (≥0.80)')
    ax.set_xlabel('Match Confidence')
    ax.set_ylabel('Companies')
    ax.legend(fontsize=8)

fig.suptitle('Level 1 XGBoost — Match Confidence by Source\n(These values already exist in Redshift match tables)',
             fontsize=13, y=1.02)
plt.tight_layout()
_save('l1_confidence_dist.png')

# Summary table
rows = []
for col, (label, _) in available_l1.items():
    v = pd.to_numeric(df[col], errors='coerce').dropna()
    hi = int((v >= 0.80).sum())
    rows.append({
        'Source':              label,
        'Mean':                f'{v.mean():.3f}',
        'Median':              f'{v.median():.3f}',
        'P25':                 f'{v.quantile(0.25):.3f}',
        'P75':                 f'{v.quantile(0.75):.3f}',
        '≥ 0.80 (Matched)':   f'{hi} ({hi/n:.0%})',
        '< 0.20 (Weak)':      f"{(v<0.20).sum()} ({(v<0.20).mean():.0%})",
        'Redshift table':      {
            'Equifax':        'efx_matches_custom_inc_ml.efx_probability',
            'OpenCorporates': 'oc_matches_custom_inc_ml.oc_probability',
            'ZoomInfo':       'zoominfo_matches_custom_inc_ml.zi_probability',
            'Liberty':        '{TABLE}_results.parquet (local batch)',
        }.get(label,''),
    })
summary_df = pd.DataFrame(rows)
display(summary_df)

In [ ]:
# ── Source confidence box plots + correlation ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Box plots
ax = axes[0]
data_box = []
labels_box = []
colours_box = []
for col, (label, colour) in available_l1.items():
    data_box.append(pd.to_numeric(df[col], errors='coerce').dropna().values)
    labels_box.append(label)
    colours_box.append(colour)
bp = ax.boxplot(data_box, labels=labels_box, patch_artist=True, widths=0.5)
for patch, colour in zip(bp['boxes'], colours_box):
    patch.set_facecolor(colour)
    patch.set_alpha(0.8)
for element in ['whiskers','caps','medians','fliers']:
    for item in bp[element]: item.set_color('white')
ax.axhline(0.80, color='white', linestyle='--', linewidth=1.2, label='0.80 threshold')
ax.set_title('Confidence Spread by Source')
ax.set_ylabel('Match Confidence')
ax.legend(fontsize=8)

# Best source distribution (which source has highest confidence per company)
ax = axes[1]
conf_cols_map = {
    'L1: EFX confidence':     'equifax',
    'L1: OC confidence':      'open_corporates',
    'L1: ZI confidence':      'zoominfo',
    'L1: Liberty confidence': 'liberty',
}
avail_conf = {v: df[k] for k, v in conf_cols_map.items() if k in df.columns}
if avail_conf:
    conf_mat = pd.DataFrame(avail_conf).apply(pd.to_numeric, errors='coerce').fillna(0)
    best_src = conf_mat.idxmax(axis=1)
    counts = best_src.value_counts()
    bars = ax.bar(counts.index, counts.values,
                  color=[SRC_C.get(s, GRAY) for s in counts.index], alpha=0.85)
    for bar, v in zip(bars, counts.values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                f'{v} ({v/n:.0%})', ha='center', fontsize=10)
    ax.set_title('Best Source per Company\n(highest Level 1 confidence among all 4 sources)')
    ax.set_ylabel('Companies')
    # Annotation
    oc_lib = counts.get('open_corporates',0) + counts.get('liberty',0)
    ax.annotate(
        f'{oc_lib} companies ({oc_lib/n:.0%}) have OC or Liberty\nas best source — production rule ignores these',
        xy=(0.5, 0.85), xycoords='axes fraction', ha='center',
        fontsize=9, color='#FC8181',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#742A2A', alpha=0.7),
    )

plt.tight_layout()
_save('l1_boxplots.png')

# NAICS accuracy when matched (if ground truth available)
naics_acc = l1_report.get('naics_accuracy_when_matched', {})
if naics_acc:
    print('\nLevel 1: NAICS accuracy when source matched (≥0.80 confidence):')
    print(f"{'Source':<16} {'Exact match':>14} {'4-digit match':>14} {'N matched':>12}")
    print('-' * 58)
    for src, stats in naics_acc.items():
        print(f"{src.upper():<16} {str(stats['top1_exact_pct'])+'%':>14} "
              f"{str(stats['top1_4digit_pct'])+'%':>14} {stats['n_matched']:>12,}")
    print()
    print('Interpretation: when Level 1 matches with high confidence, how often')
    print('does that source\'s NAICS code agree with the analyst ground truth label?')

---
## Part 2 — Production Pipeline: Current Industry Classification Output

The production rule (`customer_table.sql`) applies after Level 1:

```sql
primary_naics_code = CASE
    WHEN zi_match_confidence > efx_match_confidence  →  zi_c_naics6
    ELSE                                             →  efx_primnaicscode
END
```

**OpenCorporates and Liberty are excluded from the classification rule** even though their Level 1 
confidence scores are computed and stored. Their industry codes are never read by this rule.

**Production output:**
- ✅ Returns 1 NAICS code
- ❌ No probability / confidence measure
- ❌ UK SIC available from OC — silently dropped (no storage table)
- ❌ No AML signals
- ❌ No KYB recommendation
- ❌ OC and Liberty industry codes never used for classification

In [ ]:
# ── Production rule summary ────────────────────────────────────────────────────
prod_null  = df['Prod: NAICS code'].isna().sum() if 'Prod: NAICS code' in df.columns else 0
has_code   = n - prod_null
zi_wins    = (df['Prod: Winning source'] == 'zoominfo').sum()  if 'Prod: Winning source' in df.columns else 0
efx_wins   = (df['Prod: Winning source'] == 'equifax').sum()   if 'Prod: Winning source' in df.columns else 0
uk_avail   = (df['Prod: UK SIC returned'] != 'Not returned').sum() if 'Prod: UK SIC returned' in df.columns else 0

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Scenario A — Production Pipeline: Current Industry Classification Output',
             fontsize=12, y=1.02)

# Coverage
ax = axes[0]
ax.pie([has_code, prod_null],
       labels=[f'Has NAICS\n{has_code} ({has_code/n:.0%})',
               f'No code\n{prod_null}'],
       colors=[GREEN, RED], startangle=90, autopct='%1.0f%%',
       textprops={'color':'#E2E8F0'})
ax.set_title('NAICS Coverage')

# Winner
ax = axes[1]
ax.pie([zi_wins, efx_wins],
       labels=[f'ZoomInfo wins\n{zi_wins} ({zi_wins/n:.0%})',
               f'Equifax wins\n{efx_wins} ({efx_wins/n:.0%})'],
       colors=[GOLD, BLUE], startangle=90, autopct='%1.0f%%',
       textprops={'color':'#E2E8F0'})
ax.set_title('Production Rule Winner\n(only ZI vs EFX compared)')

# UK SIC gap
ax = axes[2]
ax.pie([0, uk_avail, n - uk_avail],
       labels=[f'Stored to DB\n(0 — no table)',
               f'Available but\nDROPPED\n{uk_avail}',
               f'Not in OC\n{n-uk_avail}'],
       colors=[GREEN, RED, GRAY], startangle=90,
       autopct=lambda p: f'{p:.0f}%' if p > 2 else '',
       textprops={'color':'#E2E8F0'})
ax.set_title('UK SIC from OpenCorporates\n(available vs stored)')

plt.tight_layout()
_save('production_summary.png')

print(f'Production output summary:')
print(f'  NAICS returned:          {has_code}/{n} ({has_code/n:.0%})')
print(f'  ZoomInfo won:            {zi_wins} ({zi_wins/n:.0%})')
print(f'  Equifax won:             {efx_wins} ({efx_wins/n:.0%})')
print(f'  UK SIC from OC:          {uk_avail} available → 0 stored')
print(f'  AML signals produced:    0')
print(f'  KYB recommendations:     0')
print(f'  Probability output:      None')

In [ ]:
# ── Production confidence vs code quality ──────────────────────────────────────
if 'Prod: Match confidence' in df.columns:
    prod_conf = pd.to_numeric(df['Prod: Match confidence'], errors='coerce').dropna()

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    ax = axes[0]
    ax.hist(prod_conf, bins=25, color=BLUE, alpha=0.85, edgecolor='#0F1117')
    ax.axvline(0.80, color='white', linestyle='--', linewidth=1.5, label='0.80 threshold')
    ax.set_title('Production Rule — Winning Source Confidence\n'
                 'Distribution (this is what drives the NAICS selection)')
    ax.set_xlabel('Match Confidence of Winning Source')
    ax.set_ylabel('Companies')
    ax.legend()
    weak = (prod_conf < 0.80).sum()
    ax.annotate(
        f'{weak} companies ({weak/n:.0%})\nhave weak winning match (<0.80)\nbut still get a NAICS code assigned',
        xy=(0.30, 0.70), xycoords='axes fraction',
        fontsize=9, color=RED,
        bbox=dict(boxstyle='round', facecolor='#5c1a1a', alpha=0.8),
    )

    ax = axes[1]
    # Compare: production winning confidence vs OC confidence (the ignored source)
    if 'L1: OC confidence' in df.columns:
        oc_conf = pd.to_numeric(df['L1: OC confidence'], errors='coerce')
        oc_better = (oc_conf > prod_conf.reindex(oc_conf.index, fill_value=0)).sum()
        better_pct = oc_better / n
        cats = ['Production winner\nbetter than OC', 'OC better than\nproduction winner']
        vals = [n - oc_better, oc_better]
        bars = ax.bar(cats, vals, color=[GREEN, RED], alpha=0.85)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                    f'{v} ({v/n:.0%})', ha='center', fontsize=11)
        ax.set_title('OC vs Production Winner Confidence\n'
                     '(OC industry codes ignored by production rule)')
        ax.set_ylabel('Companies')
        ax.annotate(
            f'For {oc_better} companies ({better_pct:.0%}), OC has a\n'
            f'STRONGER match than the production winner,\n'
            f'but OC\'s industry code is never used.',
            xy=(0.5, 0.60), xycoords='axes fraction', ha='center',
            fontsize=9, color='#FEB2B2',
            bbox=dict(boxstyle='round', facecolor='#742A2A', alpha=0.7),
        )

    plt.tight_layout()
    _save('production_confidence.png')

---
## Part 3 — Consensus Level 2 XGBoost: New Industry Classification Output

The Consensus model takes the Level 1 outputs + vendor industry codes as a **45-feature vector**
and runs `XGBClassifier(objective='multi:softprob')` to produce calibrated probabilities.

**Key difference from production:** All 4 sources contribute. OC and Liberty are
weighted features. The model also produces AML signals and KYB recommendations.

In [ ]:
# ── Level 2 training metrics ───────────────────────────────────────────────────
if eval_b:
    print('Consensus Level 2 XGBoost — Training Metrics')
    print('=' * 50)
    print(f"  Training rows:   {eval_b.get('n_train', '?'):,}")
    print(f"  Test rows:       {eval_b.get('n_test', '?'):,}")
    print(f"  NAICS classes:   {eval_b.get('n_classes', '?')}")
    print(f"  Top-1 accuracy:  {eval_b.get('top1_accuracy_pct','?')}%")
    print(f"  Top-3 accuracy:  {eval_b.get('top3_accuracy_pct','?')}%")
    print(f"  Log-loss:        {eval_b.get('log_loss','?')}")
    print()
    if eval_b.get('data_source','').upper() in ('SYNTHETIC', 'SYNTHETIC (FORCED)'):
        print('  ⚠  SYNTHETIC DATA NOTE:')
        print('     Top-1 accuracy looks low because synthetic features encode')
        print('     source confidence quality, not which industry sector a company')
        print('     is in. The model cannot learn code→label mapping without real')
        print('     vendor NAICS codes aligned to ground truth labels.')
        print()
        print('  Expected with real Redshift data + analyst labels:')
        print('     Top-1: 55–75%   Top-3: 80–90%')

# Feature importance
fi = eval_b.get('top10_features', {})
if fi:
    feat_labels = {
        'oc_matched':                   'OC matched (≥0.80) — Level 1 output',
        'efx_matched':                  'Equifax matched (≥0.80) — Level 1 output',
        'zi_matched':                   'ZoomInfo matched (≥0.80) — Level 1 output',
        'liberty_matched':              'Liberty matched (≥0.80) — Level 1 output',
        'hi_risk_sector':               'Code in AML high-risk NAICS sector',
        'source_majority_agreement':    'Sources agree on same 6-digit code',
        'web_registry_distance':        'Web vs OC registry distance (shell co.)',
        'temporal_pivot_score':         'Industry code change rate (AML signal)',
        'avg_confidence':               'Avg confidence across all 4 sources',
        'max_confidence':               'Highest single-source confidence',
        'is_naics_jurisdiction':        'Jurisdiction uses NAICS (US/CA/AU)',
        'j_us_state':                   'US state jurisdiction flag',
        'oc_confidence':                'OC weighted confidence score',
        'efx_confidence':               'EFX weighted confidence score',
        'zi_confidence':                'ZI weighted confidence score',
        'sources_above_threshold':      'Fraction of sources above 50% confidence',
        'tru_pollution_flag':           'Trulioo 4-digit SIC (data quality flag)',
    }
    fig, ax = plt.subplots(figsize=(13, 5))
    names  = [feat_labels.get(k, k) for k in fi.keys()]
    values = list(fi.values())
    clrs   = [RED if any(w in k for w in ['risk','pivot','dist','pollut','aml'])
              else GREEN for k in fi.keys()]
    ax.barh(list(range(len(names)))[::-1], values, color=clrs, alpha=0.85)
    ax.set_yticks(list(range(len(names))))
    ax.set_yticklabels(names[::-1], fontsize=10)
    ax.set_title('Feature Importances — Consensus XGBoost Level 2\n'
                 'Green = classification quality | Red = AML/risk signals')
    ax.set_xlabel('Importance Score')
    p_green = mpatches.Patch(color=GREEN, label='Classification quality features')
    p_red   = mpatches.Patch(color=RED,   label='AML / risk signal features')
    ax.legend(handles=[p_green, p_red])
    plt.tight_layout()
    _save('feature_importance.png')

In [ ]:
# ── Consensus probability distribution ────────────────────────────────────────
if 'Cons: Probability' in df.columns:
    probs = pd.to_numeric(
        df['Cons: Probability'].astype(str).str.replace('%',''),
        errors='coerce'
    ).dropna() / 100.0

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    ax = axes[0]
    ax.hist(probs, bins=25, color=GREEN, alpha=0.85, edgecolor='#0F1117')
    ax.axvline(0.70, color='#48BB78', linestyle='--', linewidth=2,
               label=f'70% — HIGH confidence ({(probs>=0.70).sum()} companies)')
    ax.axvline(0.40, color=GOLD, linestyle='--', linewidth=2,
               label=f'40% — MEDIUM confidence')
    ax.set_title('Consensus Level 2 — Probability Distribution\n'
                 '(Production never outputs a probability)')
    ax.set_xlabel('Top-1 Consensus Probability')
    ax.set_ylabel('Companies')
    ax.legend(fontsize=9)

    ax = axes[1]
    bands = {
        'HIGH (≥70%)': (probs >= 0.70).sum(),
        'MED (40–69%)': ((probs >= 0.40) & (probs < 0.70)).sum(),
        'LOW (<40%)': (probs < 0.40).sum(),
    }
    bars = ax.bar(list(bands.keys()), list(bands.values()),
                  color=[GREEN, GOLD, RED], alpha=0.85)
    for bar, (label, v) in zip(bars, bands.items()):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                f'{v} ({v/n:.0%})', ha='center', fontsize=11)
    ax.set_title('Confidence Bands')
    ax.set_ylabel('Companies')

    plt.tight_layout()
    _save('probability_dist.png')

In [ ]:
# ── KYB recommendations + AML signals ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
fig.suptitle('Consensus Level 2 — AML/KYB Outputs\n'
             '(Production generates 0 of these)', fontsize=12, y=1.02)

ax = axes[0]
if 'Cons: KYB recommendation' in df.columns:
    kyb_order  = ['APPROVE','REVIEW','ESCALATE','REJECT']
    kyb_counts = df['Cons: KYB recommendation'].value_counts()
    vals = [kyb_counts.get(k, 0) for k in kyb_order]
    bars = ax.bar(kyb_order, vals,
                  color=[KYB_C[k] for k in kyb_order], alpha=0.85)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                f'{v}\n({v/n:.0%})', ha='center', fontsize=10)
    ax.set_title('KYB Recommendation')
    ax.set_ylabel('Companies')

ax = axes[1]
if 'Cons: Risk flags' in df.columns:
    all_flags = [
        f.strip() for row in df['Cons: Risk flags']
        for f in str(row).split(';')
        if f.strip() and f.strip() != '—'
    ]
    if all_flags:
        fc = Counter(all_flags).most_common()
        labels_f  = [x[0].replace('_','\n') for x in fc]
        values_f  = [x[1] for x in fc]
        colours_f = [RED if 'RISK' in l or 'DISC' in l else GOLD for l in labels_f]
        bars = ax.barh(labels_f, values_f, color=colours_f, alpha=0.85)
        for bar, v in zip(bars, values_f):
            ax.text(bar.get_width()+0.3, bar.get_y()+bar.get_height()/2,
                    f'{v} ({v/n:.0%})', va='center', fontsize=9)
        ax.set_xlim(0, max(values_f)*1.30)
        ax.set_title('AML Signal Breakdown')
        ax.set_xlabel('Companies Flagged')
    else:
        ax.text(0.5, 0.5, 'No AML flags fired', ha='center', va='center',
                transform=ax.transAxes, color=GREEN, fontsize=13)
        ax.set_title('AML Signal Breakdown')

plt.tight_layout()
_save('aml_kyb.png')

---
## Part 4 — Side-by-Side: Production vs Consensus

In [ ]:
# ── Code agreement chart ───────────────────────────────────────────────────────
if 'Codes agree' in df.columns:
    agree    = int(df['Codes agree'].sum())
    disagree = n - agree

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    fig.suptitle('Production NAICS vs Consensus NAICS — Agreement', fontsize=12)

    ax = axes[0]
    ax.pie([agree, disagree],
           labels=[f'Same code\n{agree} ({agree/n:.0%})',
                   f'Different\n{disagree} ({disagree/n:.0%})'],
           colors=[GREEN, RED], startangle=90, autopct='%1.0f%%',
           textprops={'color':'#E2E8F0'})
    ax.set_title('Code Agreement')

    ax = axes[1]
    if 'Sector (expected)' in df.columns:
        dis_sec = df[~df['Codes agree'].astype(bool)]['Sector (expected)'].value_counts().head(8)
        ax.barh(dis_sec.index, dis_sec.values, color=RED, alpha=0.8)
        for bar, v in zip(ax.patches, dis_sec.values):
            ax.text(bar.get_width()+0.2, bar.get_y()+bar.get_height()/2,
                    str(v), va='center', fontsize=10)
        ax.set_title('Disagreements by Sector\n'
                     '(where Consensus overrides production rule)')
        ax.set_xlabel('Disagreements')

    plt.tight_layout()
    _save('code_agreement.png')
    print(f'Codes agree:    {agree} ({agree/n:.0%})')
    print(f'Codes differ:   {disagree} ({disagree/n:.0%})')
    print(f'Disagreements = cases where Consensus overrides the production rule.')

In [ ]:
# ── UK SIC gap ─────────────────────────────────────────────────────────────────
uk_avail = (df['Prod: UK SIC returned'] != 'Not returned').sum() if 'Prod: UK SIC returned' in df.columns else 0
uk_used  = (df['UK SIC: Consensus'] == '✅ Primary output').sum() if 'UK SIC: Consensus' in df.columns else 0

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle(f'UK SIC Gap: {uk_avail} companies have UK SIC in OC response', fontsize=12)

axes[0].pie([0, uk_avail, n-uk_avail],
            labels=[f'Stored (0)\nNo table exists',
                    f'Available, DROPPED\n{uk_avail}',
                    f'Not in OC\n{n-uk_avail}'],
            colors=[GREEN, RED, GRAY], startangle=90,
            autopct=lambda p: f'{p:.0f}%' if p > 2 else '',
            textprops={'color':'#E2E8F0'})
axes[0].set_title('Scenario A — Production')

axes[1].pie([uk_used, max(0, uk_avail-uk_used), n-uk_avail],
            labels=[f'Used as primary\n{uk_used}',
                    f'OC returned, routed\n{max(0,uk_avail-uk_used)}',
                    f'Not in OC\n{n-uk_avail}'],
            colors=[GREEN, GOLD, GRAY], startangle=90,
            autopct=lambda p: f'{p:.0f}%' if p > 2 else '',
            textprops={'color':'#E2E8F0'})
axes[1].set_title('Scenario B — Consensus')

plt.tight_layout()
_save('uk_sic_gap.png')
print(f'Production stored: 0 UK SIC codes')
print(f'Consensus uses:    {uk_used} UK SIC codes as primary output for GB companies')

In [ ]:
# ── Feature signals from Level 2 (what production ignores) ────────────────────
feat_plot_cols = [
    ('Cons: Majority agreement', 'Source Agreement\n(1.0 = all sources agree)', 0.80, 'high confidence'),
    ('Cons: Temporal pivot',     'Temporal Pivot\n(code change rate — AML signal)', 0.70, 'fraud signal'),
    ('Cons: Web↔Registry dist',  'Web↔Registry Distance\n(shell company signal)', 0.55, 'shell company'),
]
available_feat = [(c, l, t, tl) for c, l, t, tl in feat_plot_cols if c in df.columns]

if available_feat:
    fig, axes = plt.subplots(1, len(available_feat), figsize=(5*len(available_feat), 4))
    if len(available_feat) == 1: axes = [axes]
    fig.suptitle('Level 2 Feature Signals — Production Rule Ignores All Of These',
                 fontsize=12, y=1.02)

    for ax, (col, label, thresh, tl) in zip(axes, available_feat):
        vals = pd.to_numeric(df[col], errors='coerce').dropna()
        flagged = (vals >= thresh).sum() if col != 'Cons: Majority agreement' else (vals < thresh).sum()
        colours = [RED if (v >= thresh if col != 'Cons: Majority agreement' else v < thresh)
                   else GREEN for v in vals]
        ax.hist(vals, bins=20, color=colours[0], alpha=0.75, edgecolor='#0F1117')
        ax.axvline(thresh, color='white', linestyle='--', linewidth=1.5,
                   label=f'{thresh} — {tl}')
        ax.set_title(f'{label}\n{flagged} companies flagged')
        ax.set_xlabel('Value')
        ax.set_ylabel('Companies')
        ax.legend(fontsize=8)

    plt.tight_layout()
    _save('l2_features.png')

---
## Part 5 — Full Results Table

In [ ]:
# ── Full results table ────────────────────────────────────────────────────────
display_cols = [c for c in [
    'Company', 'Jurisdiction', 'Sector (expected)',
    'L1: EFX confidence', 'L1: OC confidence', 'L1: ZI confidence', 'L1: Liberty confidence',
    'Prod: NAICS code', 'Prod: Winning source', 'Prod: Match confidence',
    'Prod: UK SIC returned',
    'Cons: Primary code', 'Cons: Primary taxonomy', 'Cons: Probability',
    'Cons: Secondary codes', 'Cons: Majority agreement', 'Cons: Sources matched',
    'Cons: KYB recommendation', 'Cons: Risk flags',
    'UK SIC: Production', 'UK SIC: Consensus',
    'Codes agree', 'Improvement',
] if c in df.columns]

pd.set_option('display.max_colwidth', 50)
pd.set_option('display.max_columns', 30)
display(df[display_cols].head(20))

In [ ]:
# ── Improvement distribution ───────────────────────────────────────────────────
if 'Improvement' in df.columns:
    imp_counts = df['Improvement'].value_counts()
    fig, ax = plt.subplots(figsize=(10, 3))
    colours_imp = {
        '✅ Both agree': GREEN,
        '🟡 Consensus differs (check)': GOLD,
        '🔴 Production had no code': RED,
    }
    bars = ax.bar(imp_counts.index, imp_counts.values,
                  color=[colours_imp.get(k, GRAY) for k in imp_counts.index], alpha=0.85)
    for bar, v in zip(bars, imp_counts.values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                f'{v} ({v/n:.0%})', ha='center', fontsize=11)
    ax.set_title('Verdict: Production vs Consensus')
    ax.set_ylabel('Companies')
    plt.tight_layout()
    _save('improvement.png')

---
## Part 6 — Conclusions and Delta Summary

In [ ]:
# ── Summary cards ─────────────────────────────────────────────────────────────
prod_null  = df['Prod: NAICS code'].isna().sum() if 'Prod: NAICS code' in df.columns else 0
uk_avail   = (df['Prod: UK SIC returned'] != 'Not returned').sum() if 'Prod: UK SIC returned' in df.columns else 0
uk_used    = (df['UK SIC: Consensus'] == '✅ Primary output').sum() if 'UK SIC: Consensus' in df.columns else 0
aml_any    = (df['Cons: Risk flags'] != '—').sum() if 'Cons: Risk flags' in df.columns else 0
kyb_dist   = df['Cons: KYB recommendation'].value_counts().to_dict() if 'Cons: KYB recommendation' in df.columns else {}
agree_n    = int(df['Codes agree'].sum()) if 'Codes agree' in df.columns else 0

fig, axes = plt.subplots(2, 4, figsize=(20, 7))
fig.patch.set_facecolor('#0F1117')
cards = [
    (f"{n-prod_null}/{n}",   'NAICS codes returned\n(Production)',          BLUE),
    ('0',                    'Probability output\n(Production)',             RED),
    (str(uk_avail),          'UK SIC received from OC\n(Production stores 0)', RED),
    ('0',                    'AML signals produced\n(Production)',           RED),
    (f'{n}/{n}',             'NAICS codes returned\n(Consensus)',            GREEN),
    (f'{eval_b.get("top1_accuracy_pct","?")!s}%', 'Top-1 accuracy\n(Consensus)', GREEN),
    (str(int(uk_used)),      'UK SIC used as primary\n(Consensus)',         GREEN),
    (str(int(aml_any)),      'Companies with AML signal\n(Consensus)',      GOLD),
]
for ax, (val, label, colour) in zip(axes.flat, cards):
    ax.set_facecolor('#1A1F2E')
    ax.set_xlim(0,1); ax.set_ylim(0,1); ax.axis('off')
    ax.text(0.5, 0.62, val,   ha='center', va='center', fontsize=28, fontweight='bold', color=colour)
    ax.text(0.5, 0.22, label, ha='center', va='center', fontsize=10, color='#A0AEC0')
    for spine in ax.spines.values():
        spine.set_edgecolor(colour); spine.set_linewidth(2)

plt.suptitle('Production vs Consensus — Key Metrics', fontsize=14, fontweight='bold',
             color='#E2E8F0', y=1.01)
plt.tight_layout()
_save('summary_cards.png')

In [ ]:
# ── Final delta table ──────────────────────────────────────────────────────────
probs_num = pd.to_numeric(
    df.get('Cons: Probability', pd.Series()).astype(str).str.replace('%',''),
    errors='coerce'
).dropna() / 100.0

delta = [
    ('Companies processed',            str(n),                     str(n)),
    ('NAICS code returned',            f'{n-prod_null} ({(n-prod_null)/n:.0%})', f'{n} (100%)'),
    ('Calibrated probability',         '❌ None — rule only',       '✅ Yes — 0 to 100%'),
    (f'Avg probability',               '—',                        f'{probs_num.mean():.0%}' if len(probs_num) else '—'),
    ('UK SIC available from OC',       str(uk_avail),              str(uk_avail)),
    ('UK SIC stored to DB / used',     '0 — no table exists',      str(int(uk_used)) + ' — primary for GB'),
    ('AML signals produced',           '0',                        str(int(aml_any))),
    ('APPROVE',                        '— not produced',           str(kyb_dist.get('APPROVE', 0))),
    ('REVIEW',                         '— not produced',           str(kyb_dist.get('REVIEW', 0))),
    ('ESCALATE + REJECT',              '— not produced',           str(kyb_dist.get('ESCALATE',0)+kyb_dist.get('REJECT',0))),
    ('OC / Liberty industry codes used','❌ Never',                '✅ Weighted features'),
    ('Jurisdiction taxonomy routing',  '❌ Always NAICS',          '✅ NAICS/UK SIC/NACE/ISIC'),
    ('Codes agree (A = B)',            '—',                        f'{agree_n}/{n} ({agree_n/n:.0%})'),
    ('Level 1 model',                  'entity_matching v1 (same)', 'entity_matching v1 (same)'),
]
delta_df = pd.DataFrame(delta, columns=['Metric', 'Production (current)', 'Consensus (new)'])
display(delta_df)

delta_df.to_csv('data/modeling/delta_table.csv', index=False)
print('Saved: data/modeling/delta_table.csv')

In [ ]:
# ── Saved artifacts summary ────────────────────────────────────────────────────
print('=' * 60)
print('  ARTIFACTS SAVED')
print('=' * 60)
artifacts = [
    ('experiment_results.csv',             'Main results — all columns'),
    ('experiment_results.xlsx',            'Same, Excel format'),
    ('data/modeling/l1_report.json',       'Level 1 source analysis'),
    ('data/modeling/evaluation_report.json','Level 2 XGBoost metrics'),
    ('data/modeling/consensus_model.ubj',  'Trained Level 2 model binary'),
    ('data/modeling/label_encoder.pkl',    'Class index ↔ NAICS string'),
    ('data/modeling/feature_config.json',  'Feature list + params'),
    ('data/modeling/delta_table.csv',      'Production vs Consensus delta'),
    ('data/modeling/l1_confidence_dist.png','Level 1 confidence charts'),
    ('data/modeling/production_summary.png','Production rule output'),
    ('data/modeling/aml_kyb.png',          'AML/KYB signals'),
    ('data/modeling/code_agreement.png',   'Code agreement chart'),
    ('data/modeling/summary_cards.png',    'Executive summary cards'),
]
for path, desc in artifacts:
    exists = '✅' if Path(path).exists() else '⚠️ '
    print(f'  {exists}  {path:<45} {desc}')